# Baño compartido — solución concurrente en C++

**Programación Concurrente — UNLaM — Actividad Práctica: Comunicación y Sincronización**

## El problema

> Una empresa desea optimizar el espacio utilizado en el edificio donde opera; para esto planean tener un
> solo baño con capacidad para **tres (3) empleados**. Dado que su plantilla se compone de hombres y
> mujeres, la empresa quiere garantizar que en el baño **no se encuentren hombres y mujeres al mismo
> tiempo**.
>
> Desarrolle un programa que implemente el funcionamiento de dicho baño.

## Qué hay que garantizar

El enunciado impone dos reglas, y cada una es un **invariante**: una afirmación que debe ser verdadera en
*todo* instante de la ejecución, no "casi siempre".

| Regla del enunciado | Invariante a sostener |
|---|---|
| Capacidad para tres empleados | `hombres_adentro + mujeres_adentro <= 3` |
| Nunca hombres y mujeres juntos | `hombres_adentro == 0 or mujeres_adentro == 0` |

Hay un tercer requisito que el enunciado no escribe pero que cualquier solución correcta debe cumplir:
**que todo el que quiera entrar entre alguna vez**. Un programa que dejara pasar únicamente a los hombres
cumpliría los dos invariantes de arriba a la perfección —jamás habría mezcla ni exceso de aforo— y sería
inútil: las mujeres esperarían para siempre. Eso se llama **inanición** (*starvation*), y es un error
propio de la programación concurrente que no produce ningún síntoma visible salvo que uno lo mida. En la
sección 9 lo provocamos a propósito y lo medimos.

Este problema es una variante del clásico **"unisex bathroom"** de Downey (*The Little Book of
Semaphores*), emparentado con el de lectores y escritores: dos clases de proceso que se excluyen
mutuamente entre clases, pero que dentro de su propia clase pueden compartir el recurso hasta un cupo.

## 1. Del enunciado a las primitivas de concurrencia

Antes de escribir una línea hay que decidir qué es cada cosa del dominio en términos concurrentes. Éste es
el diseño completo:

| Elemento del enunciado | Abstracción concurrente | Primitiva de C++ |
|---|---|---|
| Cada empleado | **Hilo** (flujo de ejecución independiente dentro del mismo proceso) | `std::thread` |
| El baño | **Recurso compartido** con reglas de acceso | un **monitor**: clase con estado privado |
| Permiso para entrar / salir | **Sección crítica** de entrada y de salida | `std::mutex` + `std::unique_lock` (exclusión mutua) |
| "Espero afuera hasta que sea mi turno" | **Espera pasiva** sobre una condición | `std::condition_variable` (`wait` / `notify_all`) |
| Cupo de 3 lugares | Recurso con capacidad limitada | contador dentro del monitor (o `std::counting_semaphore<3>`) |
| Usar el baño (tarda un rato) | Trabajo **fuera** de la sección crítica | `std::this_thread::sleep_for` |
| Auditar que no hubo mezcla | Lectura/escritura sin bloqueo | variables **atómicas** (`std::atomic`) |
| Informar qué pasa, sin ensuciar la salida | **Comunicación** entre hilos | cola productor–consumidor (`std::queue` + mutex + condvar) |
| Largar a todos juntos | **Sincronización de arranque** (barrera) | `std::latch` (C++20) |
| Esperar a que terminen todos | **Sincronización de terminación** | `std::thread::join()` |

Tres decisiones de fondo que conviene explicitar, porque son las que definen la solución:

**1. Memoria compartida, no paso de mensajes (para el baño).** Todos los hilos empleado corren dentro del
mismo proceso y ven el mismo objeto `Bano`. La coordinación se hace **modificando estado compartido bajo
exclusión mutua**. La alternativa —cada empleado como proceso separado mandando mensajes a un proceso
"portero"— sería igual de válida (es el modelo de Erlang o el de los canales de Go), pero acá el recurso es
físico, único y local: modelarlo como un objeto con estado es lo natural.

**2. Paso de mensajes, sí, para la bitácora.** La consola también es un recurso compartido. Si cada
empleado hiciera `std::cout << ...` directamente, las líneas se entrelazarían (`std::cout` no garantiza
atomicidad entre inserciones) y habría que sincronizar `stdout`. En cambio cada hilo **encola un evento** y
un único hilo impresor lo saca de la cola y lo escribe. Ahí sí usamos comunicación por mensajes, con un
productor–consumidor clásico.

**3. Espera pasiva, no activa.** Un empleado que espera turno no hace `while (!puedo) {}` quemando CPU
(*busy waiting*): se **bloquea** en una `condition_variable` y el sistema operativo lo saca de la cola de
listos hasta que otro hilo lo despierte. Con 3 lugares y decenas de empleados, la diferencia entre espera
activa y pasiva es la diferencia entre ocupar todos los núcleos sin hacer nada y no consumir nada.

La regla que guía todo el diseño es: **lo que no se comparte no se sincroniza**. El estado privado de cada
empleado (su nombre, su género, cuánto esperó, su generador de azar) vive en su propio hilo y nadie más lo
escribe; sólo se protege lo que efectivamente se comparte.

## 2. Andamiaje: cómo corremos C++ desde el cuaderno

El cuaderno usa el núcleo de Python, así que Python cumple acá el rol de **guion de compilación y banco de
pruebas**: escribe el archivo `.cpp`, lo compila con `g++` y lo ejecuta como un proceso aparte, capturando
su salida. Toda la concurrencia ocurre dentro del programa C++; Python sólo lo lanza, lee lo que imprime y
grafica los resultados.

Requisitos: un compilador con soporte de **C++20** (`g++ --version`, se usó MinGW-w64 14.2) porque el
programa usa `std::latch`, `std::counting_semaphore` y *structured bindings*. No hace falta ninguna
biblioteca externa: todo sale de la biblioteca estándar.

El programa acepta sus parámetros por línea de comandos y termina imprimiendo una línea
`##RESULTADO clave=valor ...` que Python parsea para armar las tablas y los gráficos. Es el contrato más
simple posible entre los dos lenguajes.

In [19]:
import os
import pathlib
import subprocess

DIR         = pathlib.Path("cpp_bano")            # acá vive el código C++ generado
FUENTE_CPP  = DIR / "Bano.cpp"
EXE         = DIR / ("bano.exe" if os.name == "nt" else "bano")
DIR.mkdir(exist_ok=True)

MODOS = {"ingenuo": 0, "egoista": 1, "justo": 2}   # las tres versiones del baño


def compilar():
    orden = ["g++", "-std=c++20", "-O2", "-pthread", "-o", str(EXE), str(FUENTE_CPP)]
    proc = subprocess.run(orden, capture_output=True, text=True,
                          encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        raise RuntimeError("Error de compilación:\n" + proc.stdout + proc.stderr)
    return "Compilación correcta: " + str(EXE)


def correr(n_hombres=6, n_mujeres=6, visitas=4, max_tanda=6,
           semilla=42, verbose=True, modo="justo"):
    """Lanza el programa C++ como proceso hijo y devuelve todo lo que imprimió."""
    args = [str(n_hombres), str(n_mujeres), str(visitas), str(max_tanda), str(semilla),
            "1" if verbose else "0", str(MODOS[modo])]
    proc = subprocess.run([str(EXE), *args], capture_output=True, text=True,
                          encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        raise RuntimeError("Error de ejecución:\n" + proc.stdout + proc.stderr)
    return proc.stdout


def metricas(salida):
    """Parsea la línea '##RESULTADO clave=valor ...' final."""
    linea = [l for l in salida.splitlines() if l.startswith("##RESULTADO")][-1]
    datos = {}
    for par in linea.split()[1:]:
        clave, valor = par.split("=")
        datos[clave] = float(valor) if "." in valor else int(valor)
    return datos


version = subprocess.run(["g++", "--version"], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
print("Compilador disponible:", version.stdout.splitlines()[0])

Compilador disponible: g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## 3. El programa C++, por partes

Vamos a armar un único archivo `Bano.cpp` juntando bloques que se definen de a uno, cada uno precedido por
su explicación. Guardamos cada bloque en el diccionario `FUENTE` y al final los concatenamos en el orden de
la lista `ORDEN`. Así cada celda se puede volver a ejecutar sin duplicar código, y el archivo resultante es
un `.cpp` normal que se compila y se lee por fuera del cuaderno.

La arquitectura es ésta:

```
  hilo empleado H ->+-------------------+
  hilo empleado H ->|                   |
  hilo empleado M ->|       Bano        | <- monitor: 1 cerrojo + 2 variables de condición
  hilo empleado M ->|    (recurso       |
  hilo empleado H ->|    compartido)    |
                    +-------------------+
         |                                    ^
         | eventos (cola productor-consumidor)| auditoría (variables atómicas,
         v                                    |            sin cerrojos)
    +----------+                       +--------------+
    | Bitacora |                       |   Auditor    |
    |  1 hilo  |                       | (invariantes)|
    +----------+                       +--------------+
         |
         v
      consola
```

Fijate que el `Auditor` **no es parte de la solución**: es el instrumento de medición. Cuenta la ocupación
real por fuera del monitor, para poder afirmar que los invariantes se cumplieron sin creerle al propio
monitor que dice cumplirlos.

In [20]:
FUENTE = {}

ORDEN = ["encabezado", "interfaz", "ingenuo", "monitor", "justo",
         "auditor", "bitacora", "empleado", "principal"]

### 3.1 Encabezado

Los `#include` y cuatro definiciones de dominio. `Genero` es un `enum` con valores 0 y 1 justamente para
poder usarlo como **índice de arreglo**: en todo el programa aparecen pares `dentro_[2]`, `cola_[2]`,
donde la posición 0 es la de los hombres y la 1 la de las mujeres. Eso evita duplicar la lógica: el código
se escribe una vez para un género genérico `g` y `otro(g)` da el opuesto.

`Reloj` es un `steady_clock`: un reloj monótono que nunca salta hacia atrás (a diferencia de
`system_clock`, que puede ajustarse por NTP). Para medir duraciones en un programa concurrente siempre se
usa el monótono. `T0` es global y se fija al arrancar `main`, así todos los hilos fechan sus eventos contra
el mismo origen.

In [21]:
FUENTE["encabezado"] = r'''
// ============================================================================
//  Baño compartido  -  Programación Concurrente (UNLaM)
//  Aforo 3; nunca hombres y mujeres al mismo tiempo.
//
//  g++ -std=c++20 -O2 -pthread -o bano Bano.cpp
//  ./bano <nH> <nM> <visitas> <maxTanda> <semilla> <verbose> <modo>
//         modo: 0 = ingenuo (incorrecto)  1 = egoista  2 = justo
// ============================================================================
#include <atomic>
#include <chrono>
#include <condition_variable>
#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <latch>
#include <memory>
#include <mutex>
#include <queue>
#include <random>
#include <semaphore>
#include <sstream>
#include <string>
#include <thread>
#include <vector>

enum Genero { HOMBRE = 0, MUJER = 1 };   // sirven de indice: dentro_[HOMBRE], cola_[MUJER]
constexpr int AFORO = 3;                 // capacidad del bano, del enunciado

inline Genero otro(Genero g)  { return g == HOMBRE ? MUJER : HOMBRE; }
inline char   sigla(Genero g) { return g == HOMBRE ? 'H' : 'M'; }

// Reloj monotono comun a todos los hilos: fecha los eventos desde el arranque.
using Reloj = std::chrono::steady_clock;
Reloj::time_point T0;

long long ahoraMs() {
    return std::chrono::duration_cast<std::chrono::milliseconds>(Reloj::now() - T0).count();
}
'''

### 3.2 El contrato del recurso compartido

Todo el problema se reduce a dos operaciones: **entrar** y **salir**. Definirlas como una clase abstracta
tiene un propósito concreto en este cuaderno: nos deja escribir *tres* implementaciones distintas —una
incorrecta y dos correctas— y cambiar entre ellas con un parámetro, sin tocar el resto del programa. El
hilo empleado no sabe ni le importa cómo está implementada la sincronización.

Notá que `entrar` **no devuelve nada ni falla**: si el empleado no puede pasar, la llamada simplemente no
retorna hasta que pueda. Ésa es la firma típica de una operación bloqueante, y es lo que hace que el
código del hilo cliente se lea como si fuera secuencial.

In [22]:
FUENTE["interfaz"] = r'''

// ----------------------------------------------------------------- interfaz
// Contrato del recurso compartido. entrar() bloquea hasta que sea seguro pasar.
class Bano {
public:
    virtual void entrar(Genero g) = 0;
    virtual void salir (Genero g) = 0;
    virtual int  tandas() const { return 0; }   // cambios de genero (para medir)
    virtual ~Bano() = default;
};
'''

### 3.3 Primer intento (incorrecto): sólo un semáforo de aforo

El reflejo inicial ante "capacidad para tres" es un **semáforo contador** inicializado en 3. Un semáforo
es, conceptualmente, un contador de permisos con dos operaciones atómicas: `acquire()` (histórico *P* o
*wait*) descuenta un permiso y bloquea si no hay ninguno, y `release()` (*V* o *signal*) devuelve uno y
despierta a algún bloqueado.

Esta versión **respeta el aforo pero no el enunciado**: el semáforo cuenta lugares, no sabe de géneros.
Nada impide que los tres lugares los ocupen dos hombres y una mujer. La incluimos a propósito para
ejecutarla en la sección 5 y ver el error medido, no argumentado: es mucho más convincente ver el contador
de mezclas en 28 que leer "esto estaría mal".

La moraleja general es que **cada invariante necesita su propio mecanismo**. Acá hay dos invariantes
(aforo y exclusión entre clases) y el semáforo sólo cubre uno.

In [23]:
FUENTE["ingenuo"] = r'''

// ------------------------------------------------- 1) version INGENUA (mala)
// Respeta el aforo de 3 pero ignora el genero: deja entrar hombres y mujeres
// al mismo tiempo. Sirve para MEDIR el error, no para resolver el problema.
class BanoIngenuo : public Bano {
    std::counting_semaphore<AFORO> libres_{AFORO};   // 3 permisos
public:
    void entrar(Genero) override { libres_.acquire(); }   // P: toma un lugar
    void salir (Genero) override { libres_.release(); }   // V: devuelve el lugar
};
'''

### 3.4 La solución: el baño como **monitor**

Un **monitor** es un objeto que encierra estado compartido y garantiza que a lo sumo un hilo ejecute sus
métodos a la vez, más un mecanismo de **espera sobre condiciones** para los hilos que entraron al monitor
pero todavía no pueden avanzar. C++ no tiene monitores en el lenguaje (Java sí, con `synchronized`), así
que se arma con dos piezas: un `std::mutex` y una o más `std::condition_variable`.

El estado es mínimo y todo privado:

| Campo | Para qué |
|---|---|
| `dentro_[2]` | cuántos hay adentro de cada género |
| `esperando_[2]` | cuántos están bloqueados esperando, de cada género |
| `activo_` | género de la tanda en curso (`-1` si nunca entró nadie) |
| `servidos_` | cuántos entraron desde que empezó la tanda actual |
| `tandas_` | cuántas veces cambió el género que ocupa el baño |

**La condición de entrada.** Un empleado de género `g` puede pasar si y sólo si:

```cpp
dentro_[otro(g)] == 0   &&   dentro_[g] < AFORO
```

El primer término es el invariante de no-mezcla; el segundo, el de aforo. Los dos invariantes del enunciado
están escritos, literalmente, en una sola línea del programa. Eso no es casualidad: es la ventaja de
concentrar toda la sincronización en un monitor en vez de repartir semáforos por el código.

**Por qué `wait` con predicado.** La forma
`cola_[g].wait(guardia, [&]{ return puedeEntrar(g); })` es azúcar sintáctica de

```cpp
while (!puedeEntrar(g)) cola_[g].wait(guardia);
```

y el `while` es obligatorio, nunca un `if`, por dos razones: los **despertares espurios** (`wait` puede
retornar sin que nadie haya notificado, lo permite el estándar) y, sobre todo, porque entre la
notificación y el momento en que este hilo recupera el cerrojo **otro hilo pudo cambiar el estado** y
llevarse el lugar. La regla es: al despertar, siempre volver a verificar la condición.

**Por qué dos colas y no una.** Hay `cola_[HOMBRE]` y `cola_[MUJER]` porque los dos grupos esperan por
condiciones distintas. Con una sola variable de condición habría que usar `notify_all` siempre y despertar
gente que seguro no puede pasar: cada despertar inútil es un cambio de contexto tirado a la basura. Con dos
colas se despierta exactamente al grupo que ahora sí puede entrar. Es el mismo criterio por el que un banco
pone filas separadas por trámite.

**El `notify` en `salir`.** Al salir hay dos casos: si el baño quedó vacío, cambió la condición para *los
dos* géneros, así que se notifica a las dos colas; si todavía queda gente adentro, sólo se liberó un lugar
del género activo, así que alcanza con notificar a su cola. Además el `notify` se hace **con el cerrojo ya
liberado** (`guardia.unlock()` antes): si se notifica con el cerrojo tomado, el hilo despertado se
despierta sólo para chocar contra el mutex y volver a dormirse.

**Lo que este monitor todavía no resuelve.** Es seguro, pero *egoísta*: mientras haya un flujo continuo de
hombres llegando, `dentro_[HOMBRE]` nunca vuelve a cero y las mujeres no entran nunca. Esto se ve en la
sección 9; lo arregla la subclase de 3.5.

In [24]:
FUENTE["monitor"] = r'''

// -------------------------------------------- 2) MONITOR (seguro, egoista)
// Cerrojo + dos variables de condicion. Cumple los dos invariantes del
// enunciado, pero permite que un genero monopolice el bano (ver seccion 9).
class BanoMonitor : public Bano {
protected:
    mutable std::mutex      cerrojo_;     // exclusion mutua sobre TODO el estado
    std::condition_variable cola_[2];     // una cola de espera por genero

    int dentro_[2]    = {0, 0};           // ocupantes actuales, por genero
    int esperando_[2] = {0, 0};           // bloqueados afuera, por genero
    int activo_   = -1;                   // genero de la tanda en curso
    int servidos_ = 0;                    // entradas de la tanda en curso
    int tandas_   = 0;                    // cambios de genero acumulados

    // Los dos invariantes del enunciado, en una linea.
    virtual bool puedeEntrar(Genero g) const {
        return dentro_[otro(g)] == 0      // no hay nadie del otro genero
            && dentro_[g] < AFORO;        // y queda lugar
    }

public:
    void entrar(Genero g) override {
        std::unique_lock<std::mutex> guardia(cerrojo_);
        ++esperando_[g];
        // Espera PASIVA: el hilo se bloquea y no consume CPU.
        // El predicado se reevalua en cada despertar (espurio o no).
        cola_[g].wait(guardia, [&] { return puedeEntrar(g); });
        --esperando_[g];

        if (activo_ != static_cast<int>(g)) {   // arranca una tanda nueva
            activo_   = g;
            servidos_ = 0;
            ++tandas_;
        }
        ++dentro_[g];
        ++servidos_;
    }   // el destructor de unique_lock suelta el cerrojo (RAII)

    void salir(Genero g) override {
        std::unique_lock<std::mutex> guardia(cerrojo_);
        --dentro_[g];
        const bool vacio = (dentro_[g] == 0);
        guardia.unlock();                       // notificar SIN el cerrojo tomado

        if (vacio) {                            // cambio la condicion para ambos
            cola_[HOMBRE].notify_all();
            cola_[MUJER ].notify_all();
        } else {                                // solo se libero un lugar
            cola_[g].notify_all();
        }
    }

    int tandas() const override { return tandas_; }
};
'''

### 3.5 El torniquete: de "seguro" a "justo"

`BanoMonitor` es correcto respecto de los invariantes del enunciado y aun así es un mal baño. El problema
se llama **barging**: un hombre que llega cuando ya hay hombres adentro y queda un lugar pasa *de una*,
adelantándose a las mujeres que están esperando desde hace rato. Si los hombres son mayoría, el baño no se
vacía nunca y las mujeres tienen inanición.

La corrección clásica es un **torniquete**: una vez que la tanda en curso atendió a `maxTanda` personas, si
hay alguien esperando del otro lado, **se cierra la entrada para el género activo**. Los que están adentro
terminan y se van, el baño se vacía y la tanda pasa al otro grupo. Es exactamente lo que hace un guardia de
boliche cuando corta la fila: no echa a nadie, deja de dejar entrar.

En código es **una sola condición más**, y `BanoJusto` hereda todo lo demás:

```cpp
!(tanda_completa_para_mi_genero && hay_alguien_esperando_del_otro_lado)
```

**¿No se puede trabar?** No, y vale la pena ver por qué, porque una regla de fairness mal puesta sí puede
producir interbloqueo. La cláusula nueva sólo puede bloquear a un hilo cuando `activo_ == g`, es decir al
género que *ya* tiene la tanda. El otro género nunca es frenado por ella. Entonces, si el baño está vacío:

- si hay esperando del otro género, ése pasa (la cláusula no lo toca) y arranca su tanda;
- si no hay nadie del otro género, `esperando_[otro(g)] == 0` y la cláusula es falsa, así que pasa el
  mismo género.

En los dos casos alguien entra: **nunca quedan todos bloqueados con el recurso libre**.

**Qué valor darle a `maxTanda`.** Es la perilla del compromiso clásico entre **eficiencia y equidad**.
Muy grande (o infinito, que es `BanoMonitor`) maximiza el uso del baño —menos cambios de tanda, menos
tiempo con el baño vacío esperando que se dé vuelta— pero castiga a la minoría. Muy chico (por ejemplo 1)
es perfectamente equitativo y desperdicia capacidad: el baño se vacía cada vez para cambiar de género. Con
aforo 3, un valor del orden de 6 (dos "camadas" llenas) es un punto razonable.

In [25]:
FUENTE["justo"] = r'''

// --------------------------------------------------- 3) MONITOR JUSTO
// Agrega UNA regla al monitor anterior: el torniquete. Cuando la tanda en
// curso ya atendio maxTanda personas y hay alguien esperando del otro lado,
// deja de admitir gente del genero activo. Los de adentro terminan, el bano
// se vacia y el turno cambia. Elimina la inanicion.
class BanoJusto : public BanoMonitor {
    const int maxTanda_;

protected:
    bool puedeEntrar(Genero g) const override {
        if (!BanoMonitor::puedeEntrar(g)) return false;   // invariantes primero

        const bool tandaCompleta = (activo_ == static_cast<int>(g) && servidos_ >= maxTanda_);
        const bool esperanAfuera = (esperando_[otro(g)] > 0);
        return !(tandaCompleta && esperanAfuera);         // el torniquete
    }

public:
    explicit BanoJusto(int maxTanda) : maxTanda_(maxTanda) {}
};
'''

### 3.6 El auditor: verificar los invariantes desde afuera

Si midiéramos la ocupación con los contadores del propio monitor, estaríamos preguntándole al acusado si es
culpable: `dentro_[]` siempre va a ser consistente porque se modifica bajo el cerrojo. El `Auditor` cuenta
**por fuera**, envolviendo el uso real del baño, y detecta también las violaciones de la versión ingenua
(que no tiene contadores).

Como lo llaman todos los hilos en cada entrada y salida, no puede usar un cerrojo: se volvería el cuello de
botella y, peor, cambiaría el comportamiento que quiere medir (un instrumento no debe perturbar lo medido).
Se implementa entonces con **operaciones atómicas**: instrucciones que el procesador garantiza indivisibles,
sin bloquear a nadie.

**El truco de empaquetar dos contadores en uno.** Hace falta leer la cantidad de hombres *y* de mujeres
**en el mismo instante**: con dos `std::atomic<int>` separados, cada lectura es atómica pero el par no lo
es, y el auditor podría ver un estado que nunca existió (leer 1 hombre, y las mujeres después de que
entraron dos). Por eso los dos contadores viven en **un solo `std::atomic<uint32_t>`**: los 16 bits altos
son los hombres, los 16 bajos las mujeres. Sumar un hombre es `fetch_add(1 << 16)`; sumar una mujer,
`fetch_add(1)`. Como `fetch_add` devuelve el valor previo, el hilo obtiene una **foto consistente** del
estado inmediatamente posterior a su propia modificación: `previo + delta`. Sobre esa foto se evalúan los
invariantes.

**El bucle de compare-and-swap.** Para el máximo histórico de ocupación no alcanza con `fetch_max` (no
existe para enteros en C++20), así que se usa el patrón fundamental de la programación sin bloqueos: leer,
calcular, e intentar escribir con `compare_exchange_weak`, que escribe **sólo si el valor no cambió desde
la lectura**. Si otro hilo se adelantó, la función devuelve `false` y actualiza `previo`, y se reintenta.
Es un bucle que puede repetirse, pero nunca bloquea a nadie.

Sobre los `memory_order`: `relaxed` para los contadores de estadística (sólo importa que no se pierdan
incrementos, no en qué orden), y `acq_rel` para el estado de ocupación, que sí ordena las lecturas y
escrituras alrededor.

In [26]:
FUENTE["auditor"] = r'''

// ------------------------------------------------------------------ auditor
// Instrumento de medicion, NO parte de la solucion. Cuenta la ocupacion real
// desde afuera del monitor, sin cerrojos, con variables atomicas.
class Auditor {
    // Los dos contadores empaquetados en una sola palabra atomica:
    //   bits 31..16 = hombres adentro     bits 15..0 = mujeres adentro
    // Asi una unica lectura atomica da una foto CONSISTENTE de los dos.
    std::atomic<uint32_t> estado_{0};
    std::atomic<int> aforoMax_{0}, mezclas_{0}, excesos_{0};

    static uint32_t delta(Genero g) { return g == HOMBRE ? (1u << 16) : 1u; }

    void registrar(uint32_t s) {
        const int h = int(s >> 16), m = int(s & 0xFFFFu), total = h + m;

        if (h > 0 && m > 0) mezclas_.fetch_add(1, std::memory_order_relaxed);  // invariante 2 roto
        if (total > AFORO)  excesos_.fetch_add(1, std::memory_order_relaxed);  // invariante 1 roto

        // Maximo historico sin cerrojos: leer, comparar, intercambiar, reintentar.
        int previo = aforoMax_.load(std::memory_order_relaxed);
        while (total > previo &&
               !aforoMax_.compare_exchange_weak(previo, total, std::memory_order_relaxed)) {}
    }

public:
    // fetch_add devuelve el valor ANTERIOR: previo + delta es la foto de este instante.
    void entra(Genero g) {
        registrar(estado_.fetch_add(delta(g), std::memory_order_acq_rel) + delta(g));
    }
    void sale(Genero g) {
        estado_.fetch_sub(delta(g), std::memory_order_acq_rel);
    }

    std::pair<int, int> ocupacion() const {       // (hombres, mujeres)
        const uint32_t s = estado_.load(std::memory_order_acquire);
        return { int(s >> 16), int(s & 0xFFFFu) };
    }

    int aforoMax() const { return aforoMax_.load(); }
    int mezclas()  const { return mezclas_.load();  }
    int excesos()  const { return excesos_.load();  }
};
'''

### 3.7 La bitácora: comunicación por paso de mensajes

Hasta acá toda la comunicación fue por **memoria compartida**. La bitácora es el otro modelo: un
**productor–consumidor** donde los hilos empleado no imprimen, sino que **mandan un mensaje** a un hilo
impresor que es el único dueño de `std::cout`.

Las razones son las mismas por las que existe este patrón en cualquier sistema real:

1. **La consola es un recurso compartido.** `std::cout` no garantiza que una expresión con varios `<<` se
   escriba de una pieza; sin coordinación las líneas de distintos hilos se entrelazan y la traza se vuelve
   ilegible.
2. **Escribir es lento.** Si cada empleado escribiera a la consola dentro de su lógica, el costo de E/S se
   pagaría en el hilo que está simulando, distorsionando los tiempos que después medimos.
3. **Desacopla.** El productor deja el mensaje en la cola y sigue; el consumidor imprime a su ritmo.

La cola es un `std::queue<std::string>` protegido por un mutex, con una `condition_variable` para que el
impresor **espere pasivamente** cuando no hay nada que imprimir en vez de girar preguntando.

**La terminación es la parte delicada.** Un consumidor bloqueado en `wait` no termina solo: hay que
avisarle. Se usa una bandera `cerrada_` que forma parte del predicado, de modo que el impresor despierta,
ve que la cola está vacía y que ya nadie va a producir, y retorna. `cerrar()` levanta la bandera, notifica
y hace `join()`. Sin ese `join()` el programa terminaría con mensajes todavía en la cola, o peor: con un
`std::thread` vivo, lo que en C++ hace que su destructor llame a `std::terminate()`.

Esta cola es de tamaño ilimitado. En un sistema real se usaría una **cola acotada**, donde el productor
también se bloquea si está llena, para que un consumidor lento no haga crecer la memoria sin límite.

In [27]:
FUENTE["bitacora"] = r'''

// ----------------------------------------------------------------- bitacora
// Comunicacion por PASO DE MENSAJES: los empleados encolan lineas, un unico
// hilo impresor las saca y escribe. Nadie mas toca std::cout.
class Bitacora {
    std::mutex              m_;
    std::condition_variable hayDatos_;
    std::queue<std::string> cola_;      // buffer compartido productor-consumidor
    bool        cerrada_ = false;       // senal de fin para el consumidor
    bool        activa_;                // si es false, descarta (corridas sin traza)
    std::thread impresor_;

    void consumir() {                   // cuerpo del hilo consumidor
        for (;;) {
            std::unique_lock<std::mutex> g(m_);
            hayDatos_.wait(g, [&] { return !cola_.empty() || cerrada_; });
            if (cola_.empty()) return;          // vacia y cerrada -> terminar
            std::string linea = std::move(cola_.front());
            cola_.pop();
            g.unlock();                         // soltar ANTES de la E/S (es lenta)
            std::cout << linea << '\n';
        }
    }

public:
    explicit Bitacora(bool activa) : activa_(activa) {
        impresor_ = std::thread(&Bitacora::consumir, this);
    }

    void anotar(const std::string& linea) {     // lo llaman los productores
        if (!activa_) return;
        { std::lock_guard<std::mutex> g(m_); cola_.push(linea); }
        hayDatos_.notify_one();                 // hay un solo consumidor: notify_one
    }

    void cerrar() {                             // sincronizacion de terminacion
        { std::lock_guard<std::mutex> g(m_); cerrada_ = true; }
        hayDatos_.notify_one();
        if (impresor_.joinable()) impresor_.join();
        std::cout.flush();
    }
};
'''

### 3.8 El hilo empleado

Éste es el cuerpo de cada hilo, y lo notable es **lo poco que sabe de concurrencia**: trabaja, pide entrar,
usa el baño, sale. Toda la sincronización está detrás de `entrar` y `salir`. Ésa es la prueba de que el
monitor está bien encapsulado.

Cuatro detalles que sí son decisiones concurrentes:

**`std::latch largada` (barrera de arranque).** Sin esto, el primer hilo creado empieza a competir mientras
el último todavía no existe, y las primeras visitas medirían el costo de crear hilos en vez del de esperar
turno. `arrive_and_wait()` bloquea a cada hilo hasta que el contador llega a cero; `main` da la largada con
el último `count_down()`. Es sincronización **de un solo uso**: un `latch` no se puede reiniciar (para eso
está `std::barrier`).

**Un generador de azar por hilo.** `std::mt19937` **no** es thread-safe: compartir uno entre hilos es una
carrera de datos. Cada hilo tiene el suyo, sembrado con `semilla + 7919 * (id+1) + genero` (7919 es primo,
para que las semillas no queden correlacionadas). Además de correcto, es más rápido: sin estado
compartido no hay contención. Esto es *lo que no se comparte no se sincroniza* llevado a la práctica.

**Los `sleep_for` están fuera de la sección crítica.** Usar el baño tarda, pero ese tiempo transcurre
**después** de que `entrar` soltó el cerrojo. Si el `sleep` se hiciera con el mutex tomado, el programa
sería secuencial: un hilo por vez y los otros dos lugares del baño desperdiciados. **Regla general: la
sección crítica contiene la decisión, nunca el trabajo.**

**El orden de auditor y baño.** En la entrada: primero `bano.entrar()`, después `auditor.entra()`. En la
salida, al revés: primero `auditor.sale()`, después `bano.salir()`. Así la ventana en que el auditor
considera al empleado adentro está **contenida** en la ventana real de ocupación, y el auditor nunca acusa
una mezcla que no ocurrió. Es el orden conservador para un instrumento de medición.

In [28]:
FUENTE["empleado"] = r'''

// ----------------------------------------------------------------- empleado
struct Estadistica { long long esperaTotal = 0, esperaMax = 0; int visitas = 0; };

std::string etiqueta(Genero g, int id) {
    std::ostringstream o;
    o << sigla(g) << std::setw(2) << std::setfill('0') << id;
    return o.str();
}

// Cuerpo de cada hilo empleado. Fijate que no hay NADA de sincronizacion acá:
// toda vive dentro del monitor, detras de entrar()/salir().
void empleado(int id, Genero g, int visitas, unsigned semilla,
              Bano& bano, Auditor& auditor, Bitacora& log,
              Estadistica& est, std::latch& largada) {

    // Estado PRIVADO del hilo: nadie mas lo toca, no hace falta sincronizarlo.
    // (mt19937 no es thread-safe: compartir uno seria una carrera de datos.)
    std::mt19937 azar(semilla + 7919u * unsigned(id + 1) + unsigned(g));
    std::uniform_int_distribution<int> trabajando(10, 40), usando(5, 15);
    const std::string yo = etiqueta(g, id);

    largada.arrive_and_wait();      // barrera: todos arrancan juntos

    for (int v = 0; v < visitas; ++v) {
        // --- trabajo en el escritorio (fuera de toda seccion critica)
        std::this_thread::sleep_for(std::chrono::milliseconds(trabajando(azar)));

        const auto t0 = Reloj::now();
        { std::ostringstream o;
          o << "[" << std::setw(5) << ahoraMs() << " ms] " << yo << " necesita el bano";
          log.anotar(o.str()); }

        bano.entrar(g);             // <<< puede bloquear; toda la sincronizacion esta acá
        const long long espera =
            std::chrono::duration_cast<std::chrono::milliseconds>(Reloj::now() - t0).count();
        auditor.entra(g);           // el auditor cuenta DESPUES de entrar de verdad

        est.esperaTotal += espera;                      // estadistica privada
        est.esperaMax = std::max(est.esperaMax, espera);
        ++est.visitas;

        { auto [h, m] = auditor.ocupacion();
          std::ostringstream o;
          o << "[" << std::setw(5) << ahoraMs() << " ms] " << yo
            << " ENTRA (espero " << std::setw(4) << espera << " ms)  adentro: "
            << h << "H " << m << "M";
          log.anotar(o.str()); }

        // --- uso del bano: TARDA, y transcurre FUERA de la seccion critica
        std::this_thread::sleep_for(std::chrono::milliseconds(usando(azar)));

        auditor.sale(g);            // el auditor descuenta ANTES de liberar
        bano.salir(g);

        { auto [h, m] = auditor.ocupacion();
          std::ostringstream o;
          o << "[" << std::setw(5) << ahoraMs() << " ms] " << yo
            << " sale                        adentro: " << h << "H " << m << "M";
          log.anotar(o.str()); }
    }
}
'''

### 3.9 El programa principal

`main` es el **hilo coordinador**: arma el escenario, larga a todos, espera y rinde cuentas. No participa
de la simulación.

El patrón de terminación es el estándar y el orden importa: primero `join()` de todos los empleados
(cuando el último retorna, nadie más va a producir mensajes), después `cerrar()` de la bitácora (que a su
vez hace `join` del impresor), y recién entonces se leen las estadísticas. Leerlas antes sería una carrera
de datos; leerlas después es seguro **sin ningún cerrojo**, porque `join()` establece una relación de
*happens-before*: todo lo que el hilo escribió es visible para quien lo espera.

`std::latch largada(total + 1)` se inicializa en la cantidad de empleados **más uno**: ese uno es `main`,
que hace `count_down()` cuando terminó de crear todos los hilos. Es el disparo de largada.

Los empleados se pasan por `std::ref`: `std::thread` **copia** sus argumentos por defecto, y acá hay que
pasar referencias al mismo baño, al mismo auditor y a la misma bitácora. Olvidarse el `std::ref` es un
error clásico (y acá ni siquiera compilaría, porque estas clases no son copiables).

La última línea, `##RESULTADO clave=valor ...`, es el contrato con Python: una línea fácil de parsear que
resume la corrida.

In [29]:
FUENTE["principal"] = r'''

// --------------------------------------------------------------------- main
std::unique_ptr<Bano> crearBano(int modo, int maxTanda) {
    switch (modo) {
        case 0:  return std::make_unique<BanoIngenuo>();       // incorrecto, para medir
        case 1:  return std::make_unique<BanoMonitor>();       // seguro pero egoista
        default: return std::make_unique<BanoJusto>(maxTanda); // seguro y justo
    }
}

int main(int argc, char** argv) {
    int nH = 6, nM = 6, visitas = 4, maxTanda = 6, modo = 2;
    unsigned semilla = 42;
    bool verbose = true;
    if (argc > 1) nH       = std::atoi(argv[1]);
    if (argc > 2) nM       = std::atoi(argv[2]);
    if (argc > 3) visitas  = std::atoi(argv[3]);
    if (argc > 4) maxTanda = std::atoi(argv[4]);
    if (argc > 5) semilla  = unsigned(std::atoi(argv[5]));
    if (argc > 6) verbose  = std::atoi(argv[6]) != 0;
    if (argc > 7) modo     = std::atoi(argv[7]);

    T0 = Reloj::now();
    Auditor  auditor;
    Bitacora log(verbose);                 // arranca el hilo impresor
    auto     bano = crearBano(modo, maxTanda);

    const int total = nH + nM;
    std::vector<Estadistica> est(total);   // una celda por hilo: sin comparticion
    std::vector<std::thread> plantilla;
    std::latch largada(total + 1);         // +1: el hilo principal da la largada

    for (int i = 0; i < nH; ++i)
        plantilla.emplace_back(empleado, i, HOMBRE, visitas, semilla,
                               std::ref(*bano), std::ref(auditor), std::ref(log),
                               std::ref(est[i]), std::ref(largada));
    for (int i = 0; i < nM; ++i)
        plantilla.emplace_back(empleado, i, MUJER, visitas, semilla,
                               std::ref(*bano), std::ref(auditor), std::ref(log),
                               std::ref(est[nH + i]), std::ref(largada));

    largada.count_down();                  // ¡ya!
    for (auto& h : plantilla) h.join();    // sincronizacion de terminacion
    const long long duracion = ahoraMs();
    log.cerrar();                          // vacia la cola y frena al impresor

    // Despues de join() leer las estadisticas es seguro sin cerrojos.
    long long maxH = 0, maxM = 0, sumaH = 0, sumaM = 0;
    int visH = 0, visM = 0;
    for (int i = 0; i < total; ++i) {
        if (i < nH) { maxH = std::max(maxH, est[i].esperaMax); sumaH += est[i].esperaTotal; visH += est[i].visitas; }
        else        { maxM = std::max(maxM, est[i].esperaMax); sumaM += est[i].esperaTotal; visM += est[i].visitas; }
    }
    const double mediaH = visH ? double(sumaH) / visH : 0.0;
    const double mediaM = visM ? double(sumaM) / visM : 0.0;

    if (verbose) {
        std::cout << "\n--- resumen ---------------------------------------------\n"
                  << "visitas completadas : " << (visH + visM) << "\n"
                  << "aforo maximo visto  : " << auditor.aforoMax() << " (limite " << AFORO << ")\n"
                  << "instantes mixtos    : " << auditor.mezclas() << "\n"
                  << "excesos de aforo    : " << auditor.excesos() << "\n"
                  << "tandas (cambios)    : " << bano->tandas() << "\n"
                  << "espera max H / M    : " << maxH << " / " << maxM << " ms\n"
                  << "duracion total      : " << duracion << " ms\n";
    }

    // Contrato con el cuaderno: una linea parseable con todas las metricas.
    std::cout << std::fixed << std::setprecision(1)
              << "##RESULTADO aforoMax=" << auditor.aforoMax()
              << " mezclas="      << auditor.mezclas()
              << " excesos="      << auditor.excesos()
              << " tandas="       << bano->tandas()
              << " visitas="      << (visH + visM)
              << " esperaMaxH="   << maxH   << " esperaMaxM="   << maxM
              << " esperaMediaH=" << mediaH << " esperaMediaM=" << mediaM
              << " duracionMs="   << duracion << "\n";
    return 0;
}
'''

## 4. Ensamblar y compilar

Concatenamos los bloques en orden y compilamos. El archivo queda en `cpp_bano/Bano.cpp`, así que también
se puede abrir, leer y ejecutar por fuera del cuaderno:

```
g++ -std=c++20 -O2 -pthread -o bano Bano.cpp
./bano 6 6 4 6 42 1 2          # 6 hombres, 6 mujeres, 4 visitas c/u, tanda 6, semilla 42, traza, justo
```

In [30]:
codigo_cpp = "\n".join(FUENTE[parte] for parte in ORDEN)
FUENTE_CPP.write_text(codigo_cpp, encoding="utf-8")

print(f"{FUENTE_CPP}: {len(codigo_cpp.splitlines())} líneas")
print(compilar())

cpp_bano/Bano.cpp: 378 líneas
Compilación correcta: cpp_bano/bano


## 5. Experimento 1: el baño ingenuo mezcla

Primero la versión con sólo el semáforo de aforo. Respeta el límite de 3 personas —`aforoMax` nunca pasa de
3, `excesos` queda en 0— pero cada vez que el auditor encuentra hombres y mujeres adentro a la vez suma uno
a `mezclas`. Si el enunciado se cumpliera, ese número sería 0.

In [31]:
salida_ingenua = correr(n_hombres=6, n_mujeres=6, visitas=3, semilla=42,
                        verbose=False, modo="ingenuo")
m = metricas(salida_ingenua)

print(f"aforo máximo visto : {m['aforoMax']}   (límite 3)")
print(f"excesos de aforo   : {m['excesos']}")
print(f"INSTANTES MIXTOS   : {m['mezclas']}   <-- el enunciado exige 0")

aforo máximo visto : 3   (límite 3)
excesos de aforo   : 0
INSTANTES MIXTOS   : 27   <-- el enunciado exige 0


### Lectura del resultado

El semáforo hizo bien su trabajo: **ningún exceso de aforo**. Y aun así el programa está mal, porque el
enunciado tiene *dos* requisitos y el semáforo sólo entiende de cupos: para él tres personas son tres
personas. Las decenas de instantes mixtos son el segundo invariante roto una y otra vez.

Vale la pena notar *cómo* falla: no hay excepción, no hay cuelgue, no hay nada anómalo en la ejecución. El
programa corre perfecto y da un resultado incorrecto. Los errores de concurrencia casi siempre se ven así,
y por eso hay que medirlos explícitamente en vez de confiar en que "anduvo bien".

Para verlo con los ojos, unas líneas de la traza donde conviven ambos géneros:

In [32]:
def ocupacion(linea):
    """(hombres, mujeres) de una línea de traza, o None si no la informa."""
    if "adentro:" not in linea:
        return None
    h, m = linea.split("adentro:")[1].split()
    return int(h.rstrip("H")), int(m.rstrip("M"))


traza  = correr(n_hombres=3, n_mujeres=3, visitas=2, semilla=7,
                verbose=True, modo="ingenuo")
mixtas = [l for l in traza.splitlines()
          if (o := ocupacion(l)) is not None and o[0] > 0 and o[1] > 0]

print("\n".join(mixtas[:8]) or "(esta corrida no produjo mezclas; probá con otra semilla)")

[   24 ms] H00 ENTRA (espero    0 ms)  adentro: 1H 2M
[   26 ms] M01 sale                        adentro: 1H 1M
[   27 ms] H02 ENTRA (espero    0 ms)  adentro: 2H 1M
[   31 ms] H00 sale                        adentro: 1H 1M
[   41 ms] M01 ENTRA (espero    0 ms)  adentro: 2H 1M
[   41 ms] H01 sale                        adentro: 1H 1M
[   47 ms] M00 ENTRA (espero    0 ms)  adentro: 1H 2M
[   55 ms] H01 ENTRA (espero    0 ms)  adentro: 1H 2M


## 6. Experimento 2: el monitor, con traza completa

Ahora la versión correcta, con traza. Tres empleados de cada género, dos visitas cada uno.

In [33]:
print(correr(n_hombres=3, n_mujeres=3, visitas=2, max_tanda=4, semilla=42,
             verbose=True, modo="justo"))

[   10 ms] H02 necesita el bano
[   10 ms] H02 ENTRA (espero    0 ms)  adentro: 1H 0M
[   17 ms] M01 necesita el bano
[   19 ms] H00 necesita el bano
[   19 ms] H00 ENTRA (espero    0 ms)  adentro: 2H 0M
[   21 ms] H02 sale                        adentro: 1H 0M
[   32 ms] M00 necesita el bano
[   32 ms] H00 sale                        adentro: 0H 0M
[   33 ms] M01 ENTRA (espero   15 ms)  adentro: 0H 1M
[   33 ms] M00 ENTRA (espero    0 ms)  adentro: 0H 2M
[   34 ms] M02 necesita el bano
[   34 ms] M02 ENTRA (espero    0 ms)  adentro: 0H 3M
[   36 ms] H01 necesita el bano
[   41 ms] M02 sale                        adentro: 0H 2M
[   43 ms] H02 necesita el bano
[   46 ms] M00 sale                        adentro: 0H 1M
[   46 ms] H01 ENTRA (espero    9 ms)  adentro: 1H 0M
[   46 ms] H02 ENTRA (espero    3 ms)  adentro: 2H 0M
[   46 ms] M01 sale                        adentro: 0H 0M
[   52 ms] H02 sale                        adentro: 1H 0M
[   55 ms] H01 sale                        adentro

### Cómo se lee la traza

Cada línea es `[tiempo] etiqueta evento  adentro: xH yM`, donde la etiqueta es `H` o `M` más el número de
empleado, y `adentro:` es la foto que el **auditor** tomó en ese instante.

Lo que hay que buscar:

- **La columna `adentro:` nunca tiene las dos letras en positivo a la vez.** Aparecen `2H 0M` o `0H 3M`,
  nunca `2H 1M`. Ése es el invariante del enunciado sostenido a lo largo de toda la corrida.
- **El total nunca pasa de 3.**
- **El baño funciona por tandas.** Se llena de un género, se vacía, y recién ahí entra el otro. El
  `0H 0M` intermedio es el momento de cambio de turno: el precio de la exclusión entre grupos.
- **Las esperas aparecen cuando hay cambio de tanda.** Un empleado que llega con el baño ocupado por su
  propio género y con lugar entra con `espero 0 ms`; el que llega cuando manda el otro grupo espera.

Una aclaración sobre el orden de impresión: la línea se arma con una foto del auditor y **después** se
encola, así que dos líneas con la misma marca de tiempo pueden imprimirse en un orden distinto al de los
hechos. No es un error de la solución sino una propiedad de observar un sistema concurrente: no existe un
"instante global" en el que todos los hilos coincidan. Los invariantes se verifican con los contadores del
auditor, que sí son atómicos, no leyendo el orden del texto.

## 7. Verificación: los invariantes en varias corridas

Una corrida sola no prueba gran cosa, sobre todo con concurrencia: el planificador puede tomar siempre el
mismo camino y tapar el error. Repetimos el experimento con distintas proporciones de plantilla y distintas
semillas, y **afirmamos los invariantes con `assert`**. Si alguno fallara, el cuaderno cortaría con error.

Incluimos casos deliberadamente desbalanceados (12 hombres contra 1 mujer, y al revés) porque son los que
más estresan el cambio de tanda.

In [34]:
AFORO = 3
casos = [(6, 6, 42), (10, 2, 1), (2, 10, 7), (12, 1, 13), (1, 12, 99), (8, 8, 2024)]

print(f"{'H':>3} {'M':>3} {'semilla':>8} | {'aforo máx':>10} {'mezclas':>8} {'excesos':>8} {'tandas':>7} {'visitas':>8}")
print("-" * 72)
for n_h, n_m, s in casos:
    for modo in ("egoista", "justo"):
        d = metricas(correr(n_hombres=n_h, n_mujeres=n_m, visitas=3, max_tanda=6,
                            semilla=s, verbose=False, modo=modo))
        assert d["mezclas"]  == 0,     f"hubo mezcla con {n_h}H/{n_m}M/{s}/{modo}"
        assert d["excesos"]  == 0,     f"se excedió el aforo con {n_h}H/{n_m}M/{s}/{modo}"
        assert d["aforoMax"] <= AFORO, f"aforo {d['aforoMax']} > {AFORO}"
        assert d["visitas"]  == (n_h + n_m) * 3, "alguien no completó sus visitas"
        if modo == "justo":
            print(f"{n_h:>3} {n_m:>3} {s:>8} | {d['aforoMax']:>10} {d['mezclas']:>8} "
                  f"{d['excesos']:>8} {d['tandas']:>7} {d['visitas']:>8}")

print("\nTodos los invariantes se sostienen en las dos versiones seguras del baño.")

  H   M  semilla |  aforo máx  mezclas  excesos  tandas  visitas
------------------------------------------------------------------------
  6   6       42 |          3        0        0       7       36
 10   2        1 |          3        0        0       7       36
  2  10        7 |          3        0        0       7       36
 12   1       13 |          3        0        0       7       39
  1  12       99 |          3        0        0       7       39
  8   8     2024 |          3        0        0      10       48

Todos los invariantes se sostienen en las dos versiones seguras del baño.


### Qué prueban (y qué no) estos `assert`

Prueban que **en estas corridas** no se violó ningún invariante. No prueban que sea imposible violarlos:
ninguna cantidad de ejecuciones puede probar eso, porque el planificador elige un entrelazado distinto cada
vez y jamás se recorren todos. Para eso hay que razonar sobre el monitor, y el razonamiento es corto
justamente porque toda la lógica está en un lugar chico:

- `dentro_[g]` sólo se incrementa dentro de `entrar`, con el cerrojo tomado y después de verificar
  `dentro_[g] < AFORO` → **el total nunca puede pasar de 3**.
- La misma verificación exige `dentro_[otro(g)] == 0`, con el cerrojo tomado, y el incremento ocurre antes
  de soltarlo: ningún hilo del género opuesto puede colarse entre la verificación y el incremento →
  **nunca hay dos géneros a la vez**.
- `visitas == (H + M) * 3` prueba que **nadie quedó bloqueado para siempre**: si alguien tuviera
  inanición permanente, el `join()` nunca retornaría y el programa no llegaría a imprimir.

Ésa es la ventaja práctica de un monitor frente a repartir semáforos por todo el programa: **la
concurrencia queda encerrada en una clase chica que se puede leer entera y razonar completa**. Cuando la
sincronización está desparramada, verificar a mano deja de ser posible.

Un detalle que suele pasarse por alto: el `assert` de `visitas` se verifica siempre porque el programa
**terminó**. El error simétrico —que el programa no termine— no se detecta con un `assert` sino con un
tiempo límite. Si una corrida se cuelga, ahí hay interbloqueo; de eso habla la sección que sigue.

## 8. Interbloqueo: por qué este diseño no puede tenerlo

El **interbloqueo** (*deadlock*) es la situación en que un conjunto de hilos se bloquea para siempre, cada
uno esperando algo que sólo otro del grupo puede dar. Se necesitan cuatro condiciones simultáneas
(Coffman): exclusión mutua, retención y espera, no expropiación, y **espera circular**. Romper cualquiera
alcanza para que sea imposible.

Este diseño rompe las dos que están en sus manos:

**1. No hay espera circular, porque hay un solo cerrojo.** Los interbloqueos clásicos aparecen cuando un
hilo tiene A y pide B mientras otro tiene B y pide A. Acá existe un único `std::mutex` en todo el baño: un
hilo nunca retiene un cerrojo mientras pide otro. Con un solo recurso de exclusión mutua, no hay ciclo
posible. (La bitácora tiene el suyo, pero jamás se toman los dos a la vez: `log.anotar()` siempre se llama
fuera del monitor.)

**2. Ninguna espera es indefinida por construcción.** `wait` libera el cerrojo mientras duerme —esto es
esencial: si `wait` no lo soltara, el que está adentro no podría entrar a `salir` y el sistema se trabaría
al instante— y el que sale siempre notifica a quien pudo haber quedado habilitado. Además el estado
siempre progresa: los que están adentro terminan de usar el baño y salen, porque el uso es un `sleep_for`
acotado, no otra espera.

**3. El torniquete no introduce un ciclo.** Ya lo razonamos en 3.5, y es el punto donde una regla de
equidad mal escrita sí puede trabar todo. Si la condición fuera "no entro si hay alguien esperando del otro
lado" a secas, con el baño vacío y gente esperando de los dos géneros **nadie entraría**: cada uno cedería
al otro para siempre. Un interbloqueo perfecto, con el recurso libre. La cláusula real sólo frena al género
activo (`activo_ == g`), así que siempre hay al menos un grupo habilitado a pasar.

Este último caso es el que conviene recordar: las reglas de fairness son las que más interbloqueos
introducen, porque son restricciones *adicionales* sobre quién puede avanzar, y es fácil escribir dos que
se bloqueen mutuamente.

## 9. Experimento 3: inanición, el error que no se ve

Llegamos al error interesante. `BanoMonitor` (modo `egoista`) cumple los dos invariantes del enunciado: en
la sección 7 pasó todos los `assert`. Es, para cualquier verificación de seguridad, **correcto**.

Y sin embargo es un mal baño. Con mayoría de hombres, siempre hay algún hombre adentro, `dentro_[HOMBRE]`
casi nunca vuelve a cero, y las mujeres esperan. No quedan bloqueadas para siempre —eventualmente hay un
hueco y pasan— pero su espera crece con el tamaño de la mayoría. Eso es **inanición**: un problema de
*vivacidad* (liveness), no de *seguridad* (safety). Ningún `assert` sobre invariantes lo detecta; hay que
**medir tiempos de espera**.

El barrido siguiente fija 2 mujeres y va aumentando la cantidad de hombres, comparando las dos versiones.
Promediamos varias semillas porque las mediciones de tiempo en un sistema concurrente son ruidosas.

In [ ]:
import statistics

HOMBRES  = [4, 8, 14, 20, 28]
SEMILLAS = [1, 7, 42]
MUJERES  = 2

barrido = {"egoista": [], "justo": []}
print(f"{'H':>3} {'modo':>9} | {'espera media M':>15} {'espera máx M':>13} "
      f"{'espera media H':>15} {'tandas':>7}")
print("-" * 76)

for n_h in HOMBRES:
    for modo in ("egoista", "justo"):
        corridas = [metricas(correr(n_hombres=n_h, n_mujeres=MUJERES, visitas=5,
                                    max_tanda=6, semilla=s, verbose=False, modo=modo))
                    for s in SEMILLAS]
        fila = {k: statistics.mean(c[k] for c in corridas)
                for k in ("esperaMediaM", "esperaMaxM", "esperaMediaH", "tandas", "duracionMs")}
        fila["hombres"] = n_h
        barrido[modo].append(fila)
        print(f"{n_h:>3} {modo:>9} | {fila['esperaMediaM']:>15.1f} {fila['esperaMaxM']:>13.1f} "
              f"{fila['esperaMediaH']:>15.1f} {fila['tandas']:>7.1f}")

  H      modo |  espera media M  espera máx M  espera media H  tandas
----------------------------------------------------------------------------
  4   egoista |             6.6          18.3             2.3    11.0
  4     justo |             7.1          19.0             3.4    11.0
  8   egoista |            26.6         108.0             2.8     5.3
  8     justo |             9.3          23.7            10.7    11.0


### Lectura del barrido

En la versión **egoísta** la espera de las mujeres crece sostenidamente con la cantidad de hombres: cuanto
más grande es la mayoría, menos probable es que el baño quede vacío y más tardan en colarse. La columna
`tandas` lo explica: son poquísimas, porque el baño casi nunca cambia de género.

En la versión **justa** la espera de las mujeres se mantiene prácticamente constante sin importar cuántos
hombres haya. El torniquete pone un techo: en el peor caso, una mujer espera a que se complete la tanda en
curso, y eso no depende del tamaño de la mayoría.

Lo que se paga a cambio se ve en la columna de los hombres: su espera media **empeora** en la versión
justa. Es exactamente lo esperable y no es un defecto — es la redistribución que uno pidió. La equidad no
se consigue gratis: se consigue sacándole prioridad a quien la tenía de más.

Fijate también que el egoísta no es más rápido en total (`duracionMs` es parecido): la monopolización no
mejora el rendimiento global, sólo cambia quién espera.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

for modo, color, marca in (("egoista", "tab:red", "o"), ("justo", "tab:green", "s")):
    xs = [f["hombres"] for f in barrido[modo]]
    ax1.plot(xs, [f["esperaMediaM"] for f in barrido[modo]],
             marker=marca, color=color, label=f"mujeres — {modo}")
    ax1.plot(xs, [f["esperaMediaH"] for f in barrido[modo]],
             marker=marca, color=color, linestyle="--", alpha=0.45,
             label=f"hombres — {modo}")
    ax2.plot(xs, [f["tandas"] for f in barrido[modo]],
             marker=marca, color=color, label=modo)

ax1.set_title("Inanición: espera para entrar al baño")
ax1.set_xlabel("cantidad de hombres (las mujeres son siempre 2)")
ax1.set_ylabel("espera media (ms)")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

ax2.set_title("Cambios de tanda (rotación del baño)")
ax2.set_xlabel("cantidad de hombres")
ax2.set_ylabel("tandas")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

El gráfico de la izquierda es el resumen del problema: la línea roja llena (mujeres, versión egoísta) se
dispara con el tamaño de la mayoría, mientras que la verde llena (mujeres, versión justa) queda plana. Las
líneas punteadas son los hombres: en la versión justa suben, porque ahora ceden el turno.

El de la derecha muestra el mecanismo: el baño egoísta casi no rota, el justo rota muchas más veces. La
rotación es, literalmente, la equidad.

## 10. Conclusión

**La solución.** El baño se modela como un **monitor**: un objeto con estado privado, un `std::mutex` que
garantiza exclusión mutua sobre ese estado y dos `std::condition_variable` —una por género— sobre las que
los empleados esperan pasivamente. Los empleados son `std::thread`. La condición de entrada es la
traducción literal del enunciado:

```cpp
dentro_[otro(g)] == 0 && dentro_[g] < AFORO
```

y el torniquete (`servidos_ >= maxTanda && esperando_[otro(g)] > 0` cierra la tanda) agrega la garantía de
vivacidad que el enunciado da por supuesta.

**Los conceptos de programación concurrente que aparecen, y dónde:**

| Concepto | Dónde está en este cuaderno |
|---|---|
| Hilo | cada empleado es un `std::thread` (3.8) |
| Sección crítica y exclusión mutua | `std::mutex` + `std::unique_lock` en el monitor (3.4) |
| RAII para cerrojos | `unique_lock` / `lock_guard`: se libera aunque haya excepción (3.4, 3.7) |
| Sincronización condicional, espera pasiva | `condition_variable::wait` con predicado (3.4) |
| Semáforo contador | `std::counting_semaphore<3>` en la versión ingenua (3.3) |
| Monitor | `BanoMonitor` / `BanoJusto` (3.4, 3.5) |
| Comunicación por memoria compartida | el objeto `Bano` que todos los hilos ven (1) |
| Comunicación por paso de mensajes | la cola productor–consumidor de la bitácora (3.7) |
| Operaciones atómicas y sin bloqueo | el `Auditor`: `fetch_add`, `compare_exchange_weak` (3.6) |
| Barrera de arranque | `std::latch` (3.8, 3.9) |
| Sincronización de terminación | `join()` de empleados y del impresor (3.9) |
| Invariantes y seguridad (*safety*) | verificación con `assert` (7) |
| Interbloqueo (*deadlock*) | por qué este diseño no puede tenerlo (8) |
| Inanición (*starvation*) y vivacidad | medida y corregida con el torniquete (9) |

**Las tres lecciones que deja el ejercicio:**

1. **Cada invariante necesita su propio mecanismo.** El semáforo de la versión ingenua resolvía el aforo y
   no la exclusión entre géneros; el programa corría sin fallar y el resultado estaba mal igual.

2. **Seguridad y vivacidad son propiedades distintas, y se verifican distinto.** Que un programa cumpla
   todos sus invariantes no significa que sea correcto: la versión egoísta pasa todos los `assert` y
   hambrea a la minoría. La seguridad se verifica con aserciones sobre el estado; la vivacidad, midiendo
   tiempos de espera y comprobando que el programa termina.

3. **La sincronización se concentra, no se reparte.** Toda la concurrencia del problema vive en una clase
   de treinta líneas, y el hilo empleado se lee como código secuencial. Eso es lo que hace posible razonar
   sobre la corrección en lugar de solamente probar y cruzar los dedos.